# Stage 13A — frozen-base cache admission

This is a CPU input/provenance audit, with Internet off. It performs no model fitting, inference, balancing or accuracy evaluation. S07 remains the verified official public reference at 0.651.

**A pass admits the GLM feature cache only. Held-out predictions and a clean development reference still require separate fold-local fits.**


## Input contract

| Input | Required immutable version |
|---|---:|
| Name That LLM competition | 170484 |
| Own Stage06C feature-cache output | 359405443 |

The audit checks exact train/cache digests, safe numeric/Unicode arrays, row and text alignment, finite features, all eighteen class labels and the canonical whole-prompt folds. It never imports attached model code or deserializes object arrays.


In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys

_runner_source = '"""Stage13A: read-only admission of frozen-base GLM features, never OOF scoring.\n\nRun on CPU with Internet off. Only aggregate evidence is written. No fitting,\nprediction, Torch/custom-code import, object deserialization or balancing occurs.\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom dataclasses import dataclass\nimport hashlib\nimport json\nfrom pathlib import Path\nimport platform\nimport time\nimport zipfile\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn import __version__ as sklearn_version\nfrom sklearn.model_selection import GroupKFold\n\n\nTRAIN_SHA256 = "40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707"\nGLM_SHA256 = "2b41585a9482cef825962a8cc096697933c0b1237e8ede0712d1f8ab361d9414"\nORDERED_TEXT_SHA256 = "6496071874da8f911fafebd9eec6191e56134a97f00226dd8c78805d31f07e36"\nFOLD_SOURCE_SHA256 = "5c27fcdffde2bcfd512d20fb4edec9bd7133430e79ccd71e31875151d235a23d"\nFOLD_MAP_SHA256 = "a55b035686cb7b31f6fbe852ac990668709aec4b913c5d0928d89d843bdbfe62"\nCLASSES = (\n    "claude-fable-5", "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5",\n    "claude-sonnet-5", "gemini-3.1-pro-preview", "gemini-3.6-flash",\n    "gemini-3.7-flash", "gemini-3.8-flash", "glm-5.3", "glm-5.3-flash",\n    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra",\n    "gpt-6-astra", "kimi-k3", "qwen3.8-max",\n)\nCACHE_KEYS = {"features", "ids", "layers", "text_sha256", "token_lengths", "train_sha256"}\n\n\n@dataclass(frozen=True)\nclass Contract:\n    """The CLI always uses PRODUCTION; small contracts exist for fixtures only."""\n    train_sha256: str\n    glm_sha256: str\n    ordered_text_sha256: str\n    fold_map_sha256: str\n    rows: int = 2699\n    prompts: int = 150\n    hidden: int = 4096\n    classes: tuple[str, ...] = CLASSES\n    layers: tuple[int, ...] = (8, 3, 24)\n    folds: int = 5\n\n\nPRODUCTION = Contract(TRAIN_SHA256, GLM_SHA256, ORDERED_TEXT_SHA256, FOLD_MAP_SHA256)\n\n\nclass AdmissionFailure(Exception):\n    """Only a fixed, sanitized check code may leave the audit."""\n\n\ndef check(condition, code):\n    if not bool(condition):\n        raise AdmissionFailure(code)\n\n\ndef json_hash(value):\n    data = json.dumps(value, sort_keys=True, separators=(",", ":")).encode("utf-8")\n    return hashlib.sha256(data).hexdigest()\n\n\ndef sha256(path):\n    result = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            result.update(block)\n    return result.hexdigest()\n\n\ndef unique_file(root, name):\n    matches = [path for path in root.rglob(name) if path.is_file()]\n    check(len(matches) == 1, "unique_" + name.replace(".", "_"))\n    return matches[0]\n\n\ndef npz_headers(path, contract):\n    """Inspect object/oversize headers before NumPy can allocate or deserialize."""\n    expected_names = {key + ".npy" for key in CACHE_KEYS}\n    result = {}\n    with zipfile.ZipFile(path) as archive:\n        names = archive.namelist()\n        check(len(names) == len(set(names)) and set(names) == expected_names, "glm_cache_schema")\n        max_size = contract.rows * len(contract.layers) * contract.hidden * 4 + 2_000_000\n        check(sum(info.file_size for info in archive.infolist()) <= max_size, "glm_cache_size_bound")\n        for name in names:\n            with archive.open(name) as stream:\n                version = np.lib.format.read_magic(stream)\n                check(version in ((1, 0), (2, 0)), "npy_header_version")\n                reader = (np.lib.format.read_array_header_1_0 if version == (1, 0)\n                          else np.lib.format.read_array_header_2_0)\n                shape, fortran, dtype = reader(stream)\n            check(not dtype.hasobject, "npz_object_dtype_rejected")\n            result[name[:-4]] = (shape, dtype)\n    check(result["features"] == ((contract.rows, len(contract.layers), contract.hidden), np.dtype("float32")),\n          "glm_feature_shape_dtype")\n    for name in ("ids", "text_sha256"):\n        shape, dtype = result[name]\n        check(shape == (contract.rows,) and dtype.kind == "U" and dtype.itemsize <= 512,\n              "glm_" + name + "_header")\n    check(result["layers"] == ((len(contract.layers),), np.dtype("int64")), "glm_layers_header")\n    check(result["token_lengths"] == ((contract.rows,), np.dtype("int64")), "glm_token_lengths_header")\n    shape, dtype = result["train_sha256"]\n    check(shape == () and dtype.kind == "U" and dtype.itemsize <= 512, "glm_train_digest_header")\n    return result\n\n\ndef validate_fold_partition(groups, folds, n_folds):\n    """No prompt may cross a future fit/scoring partition; no model is fitted."""\n    groups, folds = np.asarray(groups), np.asarray(folds)\n    check(folds.shape == groups.shape and folds.dtype.kind in "iu", "fold_shape_dtype")\n    check(set(folds.tolist()) == set(range(1, n_folds + 1)), "fold_coverage")\n    mapping = {}\n    for group, fold in zip(groups, folds):\n        key = str(group)\n        check(key not in mapping or mapping[key] == int(fold), "prompt_crosses_folds")\n        mapping[key] = int(fold)\n    summaries = []\n    for number in range(1, n_folds + 1):\n        valid = folds == number\n        fit = ~valid\n        fit_groups, valid_groups = set(groups[fit]), set(groups[valid])\n        check(valid.any() and fit.any() and not (fit_groups & valid_groups), "fold_prompt_exclusion")\n        summaries.append({"fold": number, "fit_rows": int(fit.sum()), "valid_rows": int(valid.sum()),\n                          "fit_prompts": len(fit_groups), "valid_prompts": len(valid_groups),\n                          "prompt_overlap": 0})\n    return mapping, summaries\n\n\ndef audit_inputs(input_root, output_root, *, contract=PRODUCTION):\n    """Return sanitized admission evidence. Existing output directories stay intact."""\n    started = time.monotonic()\n    root, out = Path(input_root), Path(output_root)\n    production = contract == PRODUCTION\n    report = {\n        "schema_version": 1, "status": "FAILED_INPUT_ADMISSION", "contract_is_production": production,\n        "scope": "frozen_base_feature_admission_only", "fit_performed": False,\n        "model_inference_performed": False, "balancing_performed": False,\n        "held_out_predictions_produced": False, "development_reference_admitted": False,\n        "stage13b_batch_coupled_comparison_admitted": False, "full_s07_oof_admitted": False,\n        "eligible_for_future_fold_local_readout_fitting": False,\n        "official_score_estimated": False, "raw_rows_or_predictions_exported": False,\n        "object_deserialization_performed": False, "checks_passed": [],\n        "expected_input_versions": {"competition": 170484, "own_glm_output": 359405443},\n        "input_versions_verified_by_filesystem": False,\n        "runtime_versions": {"python": platform.python_version(), "numpy": np.__version__,\n                             "pandas": pd.__version__, "scikit_learn": sklearn_version},\n        "limits": [\n            "Feature admission is not fitted held-out predictions, a development score or a full-S07 reference.",\n            "Future readouts and learned preprocessing must exclude every scoring-batch prompt.",\n            "Mixed supervised source folds are not admitted for coupled balancing.",\n            "DeBERTa, adapted/full-fit S07 members and other branches are not admitted by this GLM-only audit.",\n            "Recorded cache/extractor provenance is not an independent audit of general pretrained-weight history.",\n            "Immutable input-version bindings require separate saved-UI verification.",\n            "Canonical prompt folds were previously used and remain exploratory development data.",\n        ],\n    }\n    can_write = False\n\n    def passed(code):\n        report["checks_passed"].append(code)\n\n    try:\n        check(not out.exists() or (out.is_dir() and not any(out.iterdir())), "fresh_output_directory")\n        out.mkdir(parents=True, exist_ok=True)\n        can_write = True\n        passed("fresh_output_directory")\n        check(root.is_dir(), "input_root_directory")\n        train_path, cache_path = unique_file(root, "train.csv"), unique_file(root, "glm_features.npz")\n        passed("unique_required_inputs")\n        train_digest, cache_digest = sha256(train_path), sha256(cache_path)\n        check(train_digest == contract.train_sha256, "official_train_sha256")\n        check(cache_digest == contract.glm_sha256, "glm_cache_sha256")\n        report["files"] = {"official_train": {"sha256": train_digest, "bytes": train_path.stat().st_size},\n                           "glm_cache": {"sha256": cache_digest, "bytes": cache_path.stat().st_size}}\n        passed("exact_input_file_hashes")\n        train = pd.read_csv(train_path, dtype=str)\n        check(set(train.columns) == {"id", "text", "prompt_id", "model"}, "official_train_schema")\n        check(len(train) == contract.rows and not train.isna().any().any(), "official_train_rows_nonnull")\n        check(all(train[col].str.len().gt(0).all() for col in train.columns), "official_train_nonempty_strings")\n        check(train.id.is_unique, "official_train_unique_ids")\n        ids = train.id.to_numpy(dtype=str)\n        groups = train.prompt_id.to_numpy(dtype=str)\n        classes = tuple(np.sort(train.model.unique()).tolist())\n        check(classes == contract.classes and len(np.unique(groups)) == contract.prompts, "official_classes_prompts")\n        passed("official_train_axis_contract")\n        text_hashes = np.asarray([hashlib.sha256(value.encode("utf-8")).hexdigest()\n                                  for value in train.text], dtype=str)\n        text_digest = json_hash(text_hashes.tolist())\n        check(text_digest == contract.ordered_text_sha256, "ordered_text_digest")\n        npz_headers(cache_path, contract)\n        passed("numeric_unicode_npz_headers")\n        with np.load(cache_path, allow_pickle=False) as cache:\n            check(np.array_equal(cache["ids"], ids), "glm_exact_official_id_order")\n            check(np.array_equal(cache["text_sha256"], text_hashes), "glm_exact_text_hash_order")\n            check(cache["train_sha256"].item() == contract.train_sha256, "glm_bound_train_digest")\n            check(np.array_equal(cache["layers"], contract.layers), "glm_exact_layer_order")\n            lengths = cache["token_lengths"]\n            check(bool(((lengths > 0) & (lengths <= 512)).all()), "glm_recorded_token_lengths")\n            features = cache["features"]\n            for start in range(0, contract.rows, 256):\n                check(np.isfinite(features[start:start + 256]).all(), "glm_features_finite")\n            del features\n        passed("glm_values_alignment_and_finiteness")\n        labels = {label: i for i, label in enumerate(classes)}\n        y = train.model.map(labels).to_numpy(dtype=int)\n        folds = np.full(contract.rows, -1, dtype=np.int64)\n        dummy = np.zeros((contract.rows, 1), dtype=np.uint8)\n        for number, (_, valid) in enumerate(GroupKFold(contract.folds).split(dummy, y, groups), 1):\n            folds[valid] = number\n        mapping, fold_summary = validate_fold_partition(groups, folds, contract.folds)\n        fold_hash = json_hash(mapping)\n        check(fold_hash == contract.fold_map_sha256, "canonical_prompt_fold_mapping")\n        passed("canonical_future_prompt_fold_partitions")\n        report.update(\n            status=("PASS_FROZEN_BASE_FEATURE_ADMISSION_ONLY" if production\n                    else "PASS_SYNTHETIC_FEATURE_ADMISSION_ONLY"),\n            eligible_for_future_fold_local_readout_fitting=True,\n            rows=contract.rows, prompts=contract.prompts, classes=list(classes),\n            class_supports={label: int((train.model == label).sum()) for label in classes},\n            feature_shape=[contract.rows, len(contract.layers), contract.hidden], feature_dtype="float32",\n            layers=list(contract.layers), classifier_layers=[8, 3], grouping_layer=24,\n            all_feature_values_finite=True, ordered_text_sha256=text_digest,\n            id_axis_sha256=json_hash(ids.tolist()), label_axis_sha256=json_hash(list(classes)),\n            fold_source_path="scripts/stage06c_mixed_confirmation.py", fold_source_sha256=FOLD_SOURCE_SHA256,\n            canonical_prompt_fold_mapping_sha256=fold_hash, fold_partitions=fold_summary,\n            recorded_token_length_min=int(lengths.min()), recorded_token_length_max=int(lengths.max()),\n        )\n    except AdmissionFailure as error:\n        report.update(failure_code=str(error), failure_type="AdmissionFailure")\n    except Exception as error:\n        # Never include library exceptions: they can contain row values or private paths.\n        report.update(failure_code="reader_or_environment_failure", failure_type=type(error).__name__)\n    report["checks_passed_count"] = len(report["checks_passed"])\n    report["seconds"] = round(time.monotonic() - started, 6)\n    report["report_written"] = can_write\n    if can_write:\n        try:\n            (out / "stage13a_input_admission.json").write_text(json.dumps(report, indent=2, sort_keys=True) + "\\n")\n        except Exception as error:\n            report.update(status="FAILED_INPUT_ADMISSION", failure_code="report_write_failure",\n                          failure_type=type(error).__name__, report_written=False,\n                          eligible_for_future_fold_local_readout_fitting=False)\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--input-root", default="/kaggle/input")\n    parser.add_argument("--output-root", default="/kaggle/working/stage13a_admission")\n    arguments = parser.parse_args()\n    report = audit_inputs(arguments.input_root, arguments.output_root)\n    print("STAGE13A_INPUT_ADMISSION_JSON_BEGIN", flush=True)\n    print(json.dumps(report, sort_keys=True), flush=True)\n    print("STAGE13A_INPUT_ADMISSION_JSON_END", flush=True)\n    return 0 if report["status"] == "PASS_FROZEN_BASE_FEATURE_ADMISSION_ONLY" else 2\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
_ledger_source = '{\n  "schema_version": 1,\n  "review_date": "2026-10-31",\n  "review_type": "STATIC_SOURCE_AND_RECORDED_EVIDENCE_PROVENANCE_LEDGER",\n  "repository": "Watchman77/name-that-llm",\n  "reviewed_repository_ref": "6a73f73602fb7e91dab1224bd60c61989205741f",\n  "source_reference": {\n    "path": "notebooks/Stage_12_Public_651_Reproduction.ipynb",\n    "git_freeze": "55cf8622c592690c98f67b65c3e9b6381a726b32",\n    "upstream_script_version_id": 362013743,\n    "own_production_script_version_id": 363626171\n  },\n  "scope": "No model execution, imports of vendor/custom model code, new fitting, browser actions or mounted-input validation. Source intent and recorded audit evidence are separated from checkpoint training-exclusion proof.",\n  "current_official_reference": {\n    "submission": "S07",\n    "script_version_id": 363626171,\n    "public_balanced_accuracy": "0.651",\n    "evidence_path": "results/submissions/S07_FINAL_RESULT.json"\n  },\n  "full_s07_oof_admitted": false,\n  "common_fold_scoring_reference_admitted": false,\n  "fresh_cache_admission_pending": true,\n  "reason": "No complete chain of held-out scoring-batch prompt exclusions is verified across the active production inference ensemble. Production all-official fits and averaged final/fold checkpoints cannot be labelled OOF.",\n  "status_vocabulary": {\n    "EXPLICIT_FULL_OFFICIAL_FIT": "Production source fits all official rows; zero official holdout.",\n    "MIXED_FOLD_AND_SOURCE_DECLARED_FULL_DATA_CHECKPOINTS": "Source averages fold-named artifacts and a full-data artifact for every scoring row; artifact names are not an exclusion manifest.",\n    "UNKNOWN_CHECKPOINT_TRAINING_EXCLUSION": "Pretrained/adapted state and readouts require verifiable prompt exclusions; unknown is not a pass.",\n    "NONLEARNED_BATCH_COUPLED": "No current supervised fit, but changes one row using peer-row evidence; all upstream models must exclude every scoring-batch prompt."\n  },\n  "main_branches": [\n    {\n      "id": "main.q14v5",\n      "kind": "adapted_decoder_plain_and_centered",\n      "production_active": true,\n      "source_weights": {\n        "plain_log_evidence": "3.0",\n        "centered_log_evidence_effective": "1.05",\n        "centered_multiplier_formula": "3.0 * adapt_w(1.0) * twin(0.35)"\n      },\n      "development_classification": "MIXED_FOLD_AND_SOURCE_DECLARED_FULL_DATA_CHECKPOINTS",\n      "whole_scoring_batch_prompt_exclusion_verified": false,\n      "admitted_as_full_s07_oof": false,\n      "evidence": [\n        "code cell 1: AQ14V5/AQ8V5/ALL8V5 lists",\n        "code cell 3: l24_predict averages every adapter\'s plain/centered log evidence; last adapter is explicitly called full-data",\n        "code cell 6: active MODELS"\n      ],\n      "base_model_path": "models/qwen-lm/qwen-3/transformers/14b-base/1",\n      "adapter_paths": [\n        "datasets/dhruvtibarewal/nlm-l24-adapters-v5/q14v5_f0",\n        "datasets/dhruvtibarewal/nlm-l24-adapters-v5/q14v5_f1",\n        "datasets/dhruvtibarewal/nlm-l24-adapters-v5/q14v5_f2",\n        "datasets/dhruvtibarewal/nlm-l24-adapters-v5/q14v5_f3",\n        "datasets/dhruvtibarewal/nlm-l24-adapters-v5/q14v5_f4",\n        "datasets/dhruvtibarewal/nlm-l24-adapters-v5/q14v5"\n      ],\n      "keep_layers": 27,\n      "max_length": 384,\n      "adapter_count": 6,\n      "source_declared_fold_names": 5,\n      "source_declared_full_data_names": 1,\n      "verified_adapter_training_prompt_manifests": 0,\n      "blocker": "All six adapters are averaged for each scoring row. Fold names do not establish exclusion; selecting all fold adapters plus a source-declared full-data adapter is not an OOF path.",\n      "checkpoint_source_version_mapping": "Mounted paths and identities were audited in CPU V8; this ledger does not invent a mapping from UI dataset version values to notebook metadata source IDs."\n    },\n    {\n      "id": "main.q14v5_gauss",\n      "kind": "shared_prompt_gaussian",\n      "production_active": true,\n      "source_weights": {\n        "log_evidence": "0.70",\n        "temperatures": "T1=Tg=3.0"\n      },\n      "development_classification": "EXPLICIT_FULL_OFFICIAL_FIT_WITH_FULL_DATA_ADAPTED_FEATURES",\n      "whole_scoring_batch_prompt_exclusion_verified": false,\n      "admitted_as_full_s07_oof": false,\n      "evidence": [\n        "code cell 3: train_texts branch retains features only for the last full-data adapter",\n        "code cell 3: _ga.fit(Ztr_g,y,Gtr,...)"\n      ],\n      "official_fit_rows_in_production": 2699,\n      "official_fit_prompts_in_production": 150,\n      "official_heldout_rows_in_production": 0,\n      "eligible_test_group_sizes": "2..18",\n      "blocker": "Both representation supervision and Gaussian fit must exclude scoring prompts; refitting the Gaussian alone does not remove adapter exposure."\n    },\n    {\n      "id": "main.q8v5",\n      "kind": "adapted_decoder_plain_and_centered",\n      "production_active": true,\n      "source_weights": {\n        "plain_log_evidence": "2.0",\n        "centered_log_evidence_effective": "0.70",\n        "centered_multiplier_formula": "2.0 * adapt_w(1.0) * twin(0.35)"\n      },\n      "development_classification": "MIXED_FOLD_AND_SOURCE_DECLARED_FULL_DATA_CHECKPOINTS",\n      "whole_scoring_batch_prompt_exclusion_verified": false,\n      "admitted_as_full_s07_oof": false,\n      "evidence": [\n        "code cell 1: AQ14V5/AQ8V5/ALL8V5 lists",\n        "code cell 3: l24_predict averages every adapter\'s plain/centered log evidence; last adapter is explicitly called full-data",\n        "code cell 6: active MODELS"\n      ],\n      "base_model_path": "models/qwen-lm/qwen-3/transformers/8b-base/1",\n      "adapter_paths": [\n        "datasets/dhruvtibarewal/nlm-l24-adapters-v5/q8v5_f0",\n        "datasets/dhruvtibarewal/nlm-l24-adapters-v5/q8v5_f1",\n        "datasets/dhruvtibarewal/nlm-l24-adapters-v5/q8v5_f2",\n        "datasets/dhruvtibarewal/nlm-l24-adapters-v5/q8v5_f3",\n        "datasets/dhruvtibarewal/nlm-l24-adapters-v5/q8v5_f4",\n        "datasets/dhruvtibarewal/nlm-l24-adapters-v5/q8v5"\n      ],\n      "keep_layers": 24,\n      "max_length": 384,\n      "adapter_count": 6,\n      "source_declared_fold_names": 5,\n      "source_declared_full_data_names": 1,\n      "verified_adapter_training_prompt_manifests": 0,\n      "blocker": "All six adapters are averaged for each scoring row. Fold names do not establish exclusion; selecting all fold adapters plus a source-declared full-data adapter is not an OOF path.",\n      "checkpoint_source_version_mapping": "Mounted paths and identities were audited in CPU V8; this ledger does not invent a mapping from UI dataset version values to notebook metadata source IDs."\n    },\n    {\n      "id": "main.q8v5_gauss",\n      "kind": "shared_prompt_gaussian",\n      "production_active": true,\n      "source_weights": {\n        "log_evidence": "0.44",\n        "temperatures": "T1=Tg=3.0"\n      },\n      "development_classification": "EXPLICIT_FULL_OFFICIAL_FIT_WITH_FULL_DATA_ADAPTED_FEATURES",\n      "whole_scoring_batch_prompt_exclusion_verified": false,\n      "admitted_as_full_s07_oof": false,\n      "evidence": [\n        "code cell 3: train_texts branch retains features only for the last full-data adapter",\n        "code cell 3: _ga.fit(Ztr_g,y,Gtr,...)"\n      ],\n      "official_fit_rows_in_production": 2699,\n      "official_fit_prompts_in_production": 150,\n      "official_heldout_rows_in_production": 0,\n      "eligible_test_group_sizes": "2..18",\n      "blocker": "Both representation supervision and Gaussian fit must exclude scoring prompts; refitting the Gaussian alone does not remove adapter exposure."\n    },\n    {\n      "id": "main.ll8v5",\n      "kind": "adapted_decoder_plain_and_centered",\n      "production_active": true,\n      "source_weights": {\n        "plain_log_evidence": "1.0",\n        "centered_log_evidence_effective": "0.35",\n        "centered_multiplier_formula": "1.0 * adapt_w(1.0) * twin(0.35)"\n      },\n      "development_classification": "MIXED_FOLD_AND_SOURCE_DECLARED_FULL_DATA_CHECKPOINTS",\n      "whole_scoring_batch_prompt_exclusion_verified": false,\n      "admitted_as_full_s07_oof": false,\n      "evidence": [\n        "code cell 1: AQ14V5/AQ8V5/ALL8V5 lists",\n        "code cell 3: l24_predict averages every adapter\'s plain/centered log evidence; last adapter is explicitly called full-data",\n        "code cell 6: active MODELS"\n      ],\n      "base_model_path": "models/metaresearch/llama-3.1/transformers/8b/2",\n      "adapter_paths": [\n        "datasets/dhruvtibarewal/nlm-l24-adapters-v5/ll8v5_f0",\n        "datasets/dhruvtibarewal/nlm-l24-adapters-v5/ll8v5_f1",\n        "datasets/dhruvtibarewal/nlm-l24-adapters-v5/ll8v5_f2",\n        "datasets/dhruvtibarewal/nlm-l24-adapters-v5/ll8v5_f3",\n        "datasets/dhruvtibarewal/nlm-l24-adapters-v5/ll8v5_f4",\n        "datasets/dhruvtibarewal/nlm-l24-adapters-v5/ll8v5"\n      ],\n      "keep_layers": 24,\n      "max_length": 384,\n      "adapter_count": 6,\n      "source_declared_fold_names": 5,\n      "source_declared_full_data_names": 1,\n      "verified_adapter_training_prompt_manifests": 0,\n      "blocker": "All six adapters are averaged for each scoring row. Fold names do not establish exclusion; selecting all fold adapters plus a source-declared full-data adapter is not an OOF path.",\n      "checkpoint_source_version_mapping": "Mounted paths and identities were audited in CPU V8; this ledger does not invent a mapping from UI dataset version values to notebook metadata source IDs."\n    },\n    {\n      "id": "main.n14b",\n      "kind": "adapted_decoder_plain_and_centered",\n      "production_active": true,\n      "source_weights": {\n        "plain_log_evidence": "2.5",\n        "centered_log_evidence_effective": "0.875",\n        "centered_multiplier_formula": "2.5 * adapt_w(1.0) * twin(0.35)"\n      },\n      "development_classification": "UNKNOWN_CHECKPOINT_TRAINING_EXCLUSION",\n      "whole_scoring_batch_prompt_exclusion_verified": false,\n      "admitted_as_full_s07_oof": false,\n      "evidence": [\n        "code cell 1: nmeta selectors",\n        "code cell 3: l24_predict reads fixed adapter and plain/centered heads",\n        "code cell 6: one-adapter active MODELS entry"\n      ],\n      "checkpoint_path": "datasets/nishantdahal/ntl-handwriting-blend/lora_qwen3_14b",\n      "base_model_path": "models/qwen-lm/qwen-3/transformers/14b-base/1",\n      "keep_layers": 27,\n      "max_length": 447,\n      "adapter_count": 1,\n      "verified_adapter_training_prompt_manifests": 0,\n      "blocker": "Inference does not retrain adapter or readouts. No common outer-prompt exclusion manifest is established in reviewed records."\n    },\n    {\n      "id": "main.q25_14",\n      "kind": "adapted_decoder_plain_and_centered",\n      "production_active": true,\n      "source_weights": {\n        "plain_log_evidence": "1.0",\n        "centered_log_evidence_effective": "0.35",\n        "centered_multiplier_formula": "1.0 * adapt_w(1.0) * twin(0.35)"\n      },\n      "development_classification": "UNKNOWN_CHECKPOINT_TRAINING_EXCLUSION",\n      "whole_scoring_batch_prompt_exclusion_verified": false,\n      "admitted_as_full_s07_oof": false,\n      "evidence": [\n        "code cell 1: nmeta selectors",\n        "code cell 3: l24_predict reads fixed adapter and plain/centered heads",\n        "code cell 6: one-adapter active MODELS entry"\n      ],\n      "checkpoint_path": "datasets/nishantdahal/ntl-handwriting-blend/lora_qwen25_14b",\n      "base_model_path": "models/qwen-lm/qwen2.5/transformers/14b/1",\n      "keep_layers": 32,\n      "max_length": 447,\n      "adapter_count": 1,\n      "verified_adapter_training_prompt_manifests": 0,\n      "blocker": "Inference does not retrain adapter or readouts. No common outer-prompt exclusion manifest is established in reviewed records."\n    },\n    {\n      "id": "main.deb",\n      "kind": "fine_tuned_deberta",\n      "production_active": true,\n      "source_weights": {\n        "log_evidence": "1.0"\n      },\n      "development_classification": "UNKNOWN_FINE_TUNED_ENCODER_AND_READOUT_TRAINING_EXCLUSION",\n      "whole_scoring_batch_prompt_exclusion_verified": false,\n      "admitted_as_full_s07_oof": false,\n      "evidence": [\n        "code cell 3: type enc loads encoder and head.safetensors; source calls it fully fine-tuned",\n        "code cell 6: deb active MODELS"\n      ],\n      "checkpoint_path": "datasets/nishantdahal/ntl-handwriting-blend/deberta_v3_large",\n      "max_length": 384,\n      "blocker": "This is the fine-tuned production DeBERTa, not the untouched pretrained encoder eligible for a new fold-local proxy head."\n    },\n    {\n      "id": "main.svc4",\n      "kind": "four_view_text_svc",\n      "production_active": true,\n      "source_weights": {\n        "log_evidence": "1.5"\n      },\n      "development_classification": "EXPLICIT_FULL_OFFICIAL_PLUS_EXTERNAL_POOL_FIT",\n      "whole_scoring_batch_prompt_exclusion_verified": false,\n      "admitted_as_full_s07_oof": false,\n      "evidence": [\n        "code cell 3: svc4 fits train texts plus --pool texts at runtime",\n        "code cell 6: --svc_w 1.5 --pool POOL"\n      ],\n      "official_fit_rows_in_production": 2699,\n      "official_fit_prompts_in_production": 150,\n      "official_heldout_rows_in_production": 0,\n      "feature_views": [\n        "char n-gram",\n        "word n-gram",\n        "shape",\n        "function-word"\n      ],\n      "external_training_path_selector": "**/nlm_pool_v5.parquet",\n      "external_prompt_exclusion_verified": false,\n      "blocker": "Refit vectorizers/SVC in every common outer fold and exclude external rows derived from scoring prompts. --sim removes official rows but does not automatically exclude overlapping external-pool prompts."\n    }\n  ],\n  "complement_branches": [\n    {\n      "id": "comp.stylegram",\n      "kind": "stylegram_plain_with_conditional_centered_replacement",\n      "production_active": true,\n      "source_weights": {\n        "log_evidence": "1.516"\n      },\n      "development_classification": "EXPLICIT_FULL_OFFICIAL_PLUS_EXTERNAL_FIT",\n      "whole_scoring_batch_prompt_exclusion_verified": false,\n      "admitted_as_full_s07_oof": false,\n      "evidence": [\n        "code cell 4: StyleGrams.fit(TR+EX_T); plain and centered SVC fits official + extra labels; eligible test groups replace plain predictions"\n      ],\n      "official_fit_rows_in_production": 2699,\n      "official_fit_prompts_in_production": 150,\n      "official_heldout_rows_in_production": 0,\n      "external_sources": "Six CSV filename patterns; preprocessing/deduplication/source prompts and weights are source-defined",\n      "external_prompt_exclusion_verified": false,\n      "blocker": "Rebuild text transforms and both classifiers inside common folds; public extra-prompt origins must also be excluded. Existing Stage06E stylegram OOF is a separate historical implementation/reference."\n    },\n    {\n      "id": "comp.stylometry_lgb",\n      "kind": "handcrafted_style_lgbm",\n      "production_active": true,\n      "source_weights": {\n        "log_evidence": "0.063"\n      },\n      "development_classification": "EXPLICIT_FULL_OFFICIAL_FIT",\n      "whole_scoring_batch_prompt_exclusion_verified": false,\n      "admitted_as_full_s07_oof": false,\n      "evidence": [\n        "code cell 4: F_tr from all TR; LGBMClassifier(...).fit(F_tr,y)"\n      ],\n      "official_fit_rows_in_production": 2699,\n      "official_fit_prompts_in_production": 150,\n      "official_heldout_rows_in_production": 0,\n      "blocker": "Deterministic style features can be reused; each supervised LightGBM fit requires common outer-prompt exclusions."\n    },\n    {\n      "id": "comp.glm9b",\n      "kind": "frozen_feature_encoder_with_runtime_supervised_readouts",\n      "production_active": true,\n      "source_weights": {\n        "plain_log_evidence": "0.0",\n        "centered_log_evidence": "0.165",\n        "gaussian_log_evidence": "0.165"\n      },\n      "development_classification": "EXPLICIT_FULL_OFFICIAL_READOUT_AND_GAUSSIAN_FITS",\n      "whole_scoring_batch_prompt_exclusion_verified": false,\n      "admitted_as_full_s07_oof": false,\n      "evidence": [\n        "code cell 4: LM_HEADS/ENC_HEADS, plain_and_twin and gaussian_group_head",\n        "code cell 4: scalers/LR/Gaussian fit Etr,y,PROMPT without production holdout"\n      ],\n      "model_type": "glm4",\n      "hidden_size": 4096,\n      "recorded_input_path": "notebooks/lucasboesen/name-that-llm-glm4-9b-trunc25/glm4_9b_trunc25",\n      "recorded_kernel_version_id": 358683059,\n      "plain_head_features": "L8_mean+L3_mean",\n      "gaussian_feature": "L8_mean",\n      "backbone_classification": "Source treats this as pretrained/frozen; recorded byte identity is not a retraining audit of vendor history.",\n      "official_fit_rows_in_production": 2699,\n      "official_fit_prompts_in_production": 150,\n      "official_heldout_rows_in_production": 0,\n      "conditional_group_evidence": true,\n      "zero_plain_weight_not_inactive_branch": true,\n      "blocker": "Frozen-base features may support new common-fold fits after provenance/cache admission; production readouts fit all official labels and are not OOF. Test-side standardization/centering and Gaussian groups couple rows."\n    },\n    {\n      "id": "comp.q3_8b",\n      "kind": "frozen_feature_encoder_with_runtime_supervised_readouts",\n      "production_active": true,\n      "source_weights": {\n        "plain_log_evidence": "0.0",\n        "centered_log_evidence": "0.3285",\n        "gaussian_log_evidence": "0.3285"\n      },\n      "development_classification": "EXPLICIT_FULL_OFFICIAL_READOUT_AND_GAUSSIAN_FITS",\n      "whole_scoring_batch_prompt_exclusion_verified": false,\n      "admitted_as_full_s07_oof": false,\n      "evidence": [\n        "code cell 4: LM_HEADS/ENC_HEADS, plain_and_twin and gaussian_group_head",\n        "code cell 4: scalers/LR/Gaussian fit Etr,y,PROMPT without production holdout"\n      ],\n      "model_type": "qwen3",\n      "hidden_size": 4096,\n      "recorded_input_path": "models/qwen-lm/qwen-3/transformers/8b-base/1",\n      "recorded_kernel_version_id": null,\n      "plain_head_features": "L7_mean+L22_mean",\n      "gaussian_feature": "L22_mean",\n      "backbone_classification": "Source treats this as pretrained/frozen; recorded byte identity is not a retraining audit of vendor history.",\n      "official_fit_rows_in_production": 2699,\n      "official_fit_prompts_in_production": 150,\n      "official_heldout_rows_in_production": 0,\n      "conditional_group_evidence": true,\n      "zero_plain_weight_not_inactive_branch": true,\n      "blocker": "Frozen-base features may support new common-fold fits after provenance/cache admission; production readouts fit all official labels and are not OOF. Test-side standardization/centering and Gaussian groups couple rows."\n    },\n    {\n      "id": "comp.q25_7b",\n      "kind": "frozen_feature_encoder_with_runtime_supervised_readouts",\n      "production_active": true,\n      "source_weights": {\n        "plain_log_evidence": "0.0",\n        "centered_log_evidence": "0.2115"\n      },\n      "development_classification": "EXPLICIT_FULL_OFFICIAL_READOUT_AND_GAUSSIAN_FITS",\n      "whole_scoring_batch_prompt_exclusion_verified": false,\n      "admitted_as_full_s07_oof": false,\n      "evidence": [\n        "code cell 4: LM_HEADS/ENC_HEADS, plain_and_twin and gaussian_group_head",\n        "code cell 4: scalers/LR/Gaussian fit Etr,y,PROMPT without production holdout"\n      ],\n      "model_type": "qwen2",\n      "hidden_size": 3584,\n      "recorded_input_path": "models/qwen-lm/qwen2.5/transformers/7b/1",\n      "recorded_kernel_version_id": null,\n      "plain_head_features": "L6_mean+L20_mean",\n      "gaussian_feature": null,\n      "backbone_classification": "Source treats this as pretrained/frozen; recorded byte identity is not a retraining audit of vendor history.",\n      "official_fit_rows_in_production": 2699,\n      "official_fit_prompts_in_production": 150,\n      "official_heldout_rows_in_production": 0,\n      "conditional_group_evidence": true,\n      "zero_plain_weight_not_inactive_branch": true,\n      "blocker": "Frozen-base features may support new common-fold fits after provenance/cache admission; production readouts fit all official labels and are not OOF. Test-side standardization/centering and Gaussian groups couple rows."\n    },\n    {\n      "id": "comp.q3_4b",\n      "kind": "frozen_feature_encoder_with_runtime_supervised_readouts",\n      "production_active": true,\n      "source_weights": {\n        "plain_log_evidence": "0.0",\n        "centered_log_evidence": "0.1245",\n        "gaussian_log_evidence": "0.1245"\n      },\n      "development_classification": "EXPLICIT_FULL_OFFICIAL_READOUT_AND_GAUSSIAN_FITS",\n      "whole_scoring_batch_prompt_exclusion_verified": false,\n      "admitted_as_full_s07_oof": false,\n      "evidence": [\n        "code cell 4: LM_HEADS/ENC_HEADS, plain_and_twin and gaussian_group_head",\n        "code cell 4: scalers/LR/Gaussian fit Etr,y,PROMPT without production holdout"\n      ],\n      "model_type": "qwen3",\n      "hidden_size": 2560,\n      "recorded_input_path": "models/qwen-lm/qwen-3/transformers/4b-base/1",\n      "recorded_kernel_version_id": null,\n      "plain_head_features": "L4_style+L7_style",\n      "gaussian_feature": "L4_mean",\n      "backbone_classification": "Source treats this as pretrained/frozen; recorded byte identity is not a retraining audit of vendor history.",\n      "official_fit_rows_in_production": 2699,\n      "official_fit_prompts_in_production": 150,\n      "official_heldout_rows_in_production": 0,\n      "conditional_group_evidence": true,\n      "zero_plain_weight_not_inactive_branch": true,\n      "blocker": "Frozen-base features may support new common-fold fits after provenance/cache admission; production readouts fit all official labels and are not OOF. Test-side standardization/centering and Gaussian groups couple rows."\n    },\n    {\n      "id": "comp.debv3l",\n      "kind": "frozen_feature_encoder_with_runtime_supervised_readouts",\n      "production_active": true,\n      "source_weights": {\n        "plain_log_evidence": "0.05",\n        "centered_log_evidence": "0.25",\n        "gaussian_log_evidence": "0.25"\n      },\n      "development_classification": "EXPLICIT_FULL_OFFICIAL_READOUT_AND_GAUSSIAN_FITS",\n      "whole_scoring_batch_prompt_exclusion_verified": false,\n      "admitted_as_full_s07_oof": false,\n      "evidence": [\n        "code cell 4: LM_HEADS/ENC_HEADS, plain_and_twin and gaussian_group_head",\n        "code cell 4: scalers/LR/Gaussian fit Etr,y,PROMPT without production holdout"\n      ],\n      "model_type": "deberta-v2",\n      "hidden_size": 1024,\n      "recorded_input_path": "notebooks/lucasboesen/name-that-llm-extract-encoders2/deberta-v3-large",\n      "recorded_kernel_version_id": 358745836,\n      "plain_head_features": "L24_mean+L6_mean",\n      "gaussian_feature": "L24_mean",\n      "backbone_classification": "Source treats this as pretrained/frozen; recorded byte identity is not a retraining audit of vendor history.",\n      "official_fit_rows_in_production": 2699,\n      "official_fit_prompts_in_production": 150,\n      "official_heldout_rows_in_production": 0,\n      "conditional_group_evidence": true,\n      "zero_plain_weight_not_inactive_branch": false,\n      "blocker": "Frozen-base features may support new common-fold fits after provenance/cache admission; production readouts fit all official labels and are not OOF. Test-side standardization/centering and Gaussian groups couple rows."\n    },\n    {\n      "id": "comp.electl",\n      "kind": "frozen_feature_encoder_with_runtime_supervised_readouts",\n      "production_active": true,\n      "source_weights": {\n        "plain_log_evidence": "0.032"\n      },\n      "development_classification": "EXPLICIT_FULL_OFFICIAL_READOUT_AND_GAUSSIAN_FITS",\n      "whole_scoring_batch_prompt_exclusion_verified": false,\n      "admitted_as_full_s07_oof": false,\n      "evidence": [\n        "code cell 4: LM_HEADS/ENC_HEADS, plain_and_twin and gaussian_group_head",\n        "code cell 4: scalers/LR/Gaussian fit Etr,y,PROMPT without production holdout"\n      ],\n      "model_type": "electra",\n      "hidden_size": 1024,\n      "recorded_input_path": "notebooks/lucasboesen/name-that-llm-electra-large-weights/electra-large-discriminator",\n      "recorded_kernel_version_id": 358758062,\n      "plain_head_features": "L21_mean+L24_style",\n      "gaussian_feature": null,\n      "backbone_classification": "Source treats this as pretrained/frozen; recorded byte identity is not a retraining audit of vendor history.",\n      "official_fit_rows_in_production": 2699,\n      "official_fit_prompts_in_production": 150,\n      "official_heldout_rows_in_production": 0,\n      "conditional_group_evidence": false,\n      "zero_plain_weight_not_inactive_branch": false,\n      "blocker": "Frozen-base features may support new common-fold fits after provenance/cache admission; production readouts fit all official labels and are not OOF. Test-side standardization/centering and Gaussian groups couple rows."\n    },\n    {\n      "id": "comp.desk",\n      "kind": "frozen_feature_encoder_with_runtime_supervised_readouts",\n      "production_active": true,\n      "source_weights": {\n        "plain_log_evidence": "0.132"\n      },\n      "development_classification": "EXPLICIT_FULL_OFFICIAL_READOUT_AND_GAUSSIAN_FITS",\n      "whole_scoring_batch_prompt_exclusion_verified": false,\n      "admitted_as_full_s07_oof": false,\n      "evidence": [\n        "code cell 4: LM_HEADS/ENC_HEADS, plain_and_twin and gaussian_group_head",\n        "code cell 4: scalers/LR/Gaussian fit Etr,y,PROMPT without production holdout"\n      ],\n      "model_type": "deberta-v2",\n      "hidden_size": 1024,\n      "recorded_input_path": "notebooks/lucasboesen/name-that-llm-desklib-weights/desklib-detector",\n      "recorded_kernel_version_id": 358770408,\n      "plain_head_features": "L23_mean+L23_style",\n      "gaussian_feature": null,\n      "backbone_classification": "Source treats this as pretrained/frozen; recorded byte identity is not a retraining audit of vendor history.",\n      "official_fit_rows_in_production": 2699,\n      "official_fit_prompts_in_production": 150,\n      "official_heldout_rows_in_production": 0,\n      "conditional_group_evidence": false,\n      "zero_plain_weight_not_inactive_branch": false,\n      "blocker": "Frozen-base features may support new common-fold fits after provenance/cache admission; production readouts fit all official labels and are not OOF. Test-side standardization/centering and Gaussian groups couple rows."\n    },\n    {\n      "id": "comp.lora24_seed7",\n      "kind": "fixed_final_adapted_checkpoint_and_readouts",\n      "production_active": true,\n      "source_weights": {\n        "plain_log_evidence": "0.113"\n      },\n      "development_classification": "UNKNOWN_FINAL_CHECKPOINT_TRAINING_EXCLUSION",\n      "whole_scoring_batch_prompt_exclusion_verified": false,\n      "admitted_as_full_s07_oof": false,\n      "evidence": [\n        "code cell 4: LORA_RUNS and score_lora_checkpoints; job_final/final adapter plus readouts.npz; not outer-checkpoint routing"\n      ],\n      "checkpoint_path": "notebooks/lucasboesen/name-that-llm-train-lora24-s7/lora_out/job_final",\n      "checkpoint_selector": null,\n      "recorded_kernel_version_id": 358911769,\n      "blocker": "Separate historical held-out outputs, where available, do not make the loaded final production adapter held-out. Need verified exclusions for the actual backbone, norms and every readout and all prompts sharing a scoring batch."\n    },\n    {\n      "id": "comp.lora24_claude",\n      "kind": "fixed_final_adapted_checkpoint_and_readouts",\n      "production_active": true,\n      "source_weights": {\n        "plain_log_evidence": "0.177",\n        "centered_log_evidence": "0.15"\n      },\n      "development_classification": "UNKNOWN_FINAL_CHECKPOINT_TRAINING_EXCLUSION",\n      "whole_scoring_batch_prompt_exclusion_verified": false,\n      "admitted_as_full_s07_oof": false,\n      "evidence": [\n        "code cell 4: LORA_RUNS and score_lora_checkpoints; job_final/final adapter plus readouts.npz; not outer-checkpoint routing"\n      ],\n      "checkpoint_path": "datasets/zbigniewkirchner/ntl-qwen3-8b-lora-claude-head/final",\n      "checkpoint_selector": "**/ntl-qwen3-8b-lora-claude-head/**/final/COMPLETE.json",\n      "recorded_kernel_version_id": null,\n      "blocker": "Separate historical held-out outputs, where available, do not make the loaded final production adapter held-out. Need verified exclusions for the actual backbone, norms and every readout and all prompts sharing a scoring batch."\n    },\n    {\n      "id": "comp.lora20_q25_instyle",\n      "kind": "fixed_final_adapted_checkpoint_and_readouts",\n      "production_active": true,\n      "source_weights": {\n        "plain_log_evidence": "0.170"\n      },\n      "development_classification": "UNKNOWN_FINAL_CHECKPOINT_TRAINING_EXCLUSION",\n      "whole_scoring_batch_prompt_exclusion_verified": false,\n      "admitted_as_full_s07_oof": false,\n      "evidence": [\n        "code cell 4: LORA_RUNS and score_lora_checkpoints; job_final/final adapter plus readouts.npz; not outer-checkpoint routing"\n      ],\n      "checkpoint_path": "notebooks/vnton37nguyn/ntl-train-lora-qwen25-instyle/lora_out/job_final",\n      "checkpoint_selector": null,\n      "recorded_kernel_version_id": 359652568,\n      "blocker": "Separate historical held-out outputs, where available, do not make the loaded final production adapter held-out. Need verified exclusions for the actual backbone, norms and every readout and all prompts sharing a scoring batch."\n    },\n    {\n      "id": "comp.tokattn_deb",\n      "kind": "fixed_final_supervised_token_attention_head",\n      "production_active": true,\n      "source_weights": {\n        "log_evidence": "0.185"\n      },\n      "development_classification": "UNKNOWN_FINAL_HEAD_TRAINING_EXCLUSION",\n      "whole_scoring_batch_prompt_exclusion_verified": false,\n      "admitted_as_full_s07_oof": false,\n      "evidence": [\n        "code cell 4: TOKATTN_HEADS and token_attention_probs; TINY comment explicitly calls training-row test in-sample"\n      ],\n      "checkpoint_path": "notebooks/lucasboesen/name-that-llm-train-tokattn2/tokattn_debl23_final.pt",\n      "recorded_kernel_version_id": 358948428,\n      "recorded_checkpoint_sha256": "ed384fd1af244ad1e9dca42194e423ad7fc2cce7e1926820344bffec6de3c561",\n      "separate_historical_oof_npz_available": true,\n      "blocker": "Historical tokattn_debl23_oof.npz has source seed0 prompt-disjoint evidence, but the final .pt used by S07 is not those held-out predictions; arbitrary pooling with other fold schemes cannot admit coupled balancing."\n    },\n    {\n      "id": "comp.ultrabert",\n      "kind": "five_checkpoint_supervised_encoder_average",\n      "production_active": true,\n      "source_weights": {\n        "log_evidence": "0.095"\n      },\n      "development_classification": "UNKNOWN_OUTER_PROMPT_EXCLUSION_FOR_FIVE_FOLD_AVERAGE",\n      "whole_scoring_batch_prompt_exclusion_verified": false,\n      "admitted_as_full_s07_oof": false,\n      "evidence": [\n        "code cell 4: ultrabert_probs loads fold0..fold4 and averages all predictions for every scoring row",\n        "results/stage_06g/REPRESENTATION_COVERAGE.md: fold provenance/OOF unresolved"\n      ],\n      "checkpoint_path": "datasets/pjmathematician/ultrabert-380m-ntl",\n      "dataset_display_name": "UltraBERT-380M Name That LLM 5-fold",\n      "recorded_ui_dataset_version": 1,\n      "recorded_ui_version_value": "21895589",\n      "checkpoint_count": 5,\n      "blocker": "Fold directory names/configs are not training-prompt manifests. Averaging all folds is an inference ensemble, not OOF."\n    }\n  ],\n  "auxiliary_and_inactive_paths": [\n    {\n      "id": "comp.family_gates",\n      "production_scoring_contribution": "NOT_CONSUMED_BY_SAVED_LOGIT_IN_REVIEWED_CODE",\n      "fits": "Family style classifiers fit official+extra family-labelled rows; Claude LoRA gate also computed.",\n      "source": "code cell 4: GATES assignments; saved output is LOGIT written immediately after token attention/UltraBERT",\n      "exclusion": "Do not count comments about re-ranking as an active score path. Any future activation is a new candidate with its own fold-local/external exclusion requirements."\n    },\n    {\n      "id": "comp.BALANCE_ALPHA",\n      "production_scoring_contribution": "DECLARED_NOT_CONSUMED_IN_SAVED_LOGIT_PATH",\n      "value": "0.5",\n      "source": "code cell 4 constant versus final np.save(LOGIT)"\n    },\n    {\n      "id": "comp.lora24_public",\n      "production_scoring_contribution": "PREDECLARED_ALLOWED_SKIP",\n      "configured_plain_log_weight": "0.177",\n      "configured_centered_log_weight": "0.15",\n      "selector": "**/lora24/final/COMPLETE.json",\n      "source": "code cell 4 LORA_RUNS; absence confirmed in own preview",\n      "exclusion": "Not an active S07 member; the separate historical Stage06A original public LoRA OOF bank cannot substitute silently."\n    },\n    {\n      "id": "comp.tokattn_q35",\n      "production_scoring_contribution": "INTENTIONALLY_DISABLED",\n      "configured_log_weight": "0.153",\n      "control": "NTL_DROP_Q35=1",\n      "source": "code cell 5 sets environment; code cell 4 filters TOKATTN_HEADS",\n      "exclusion": "Not an active S07 member; enabling it is a changed candidate."\n    }\n  ],\n  "atlas_branch": {\n    "id": "atlas",\n    "kind": "source_conditioned_likelihood_adapter",\n    "production_active": true,\n    "source_weights": {\n      "probability_mix_weight": "0.10",\n      "selection": "top-two margin <0.3 when N>=50; all rows for smaller input"\n    },\n    "development_classification": "UNKNOWN_BUNDLE_TRAINING_PROMPT_EXCLUSION",\n    "whole_scoring_batch_prompt_exclusion_verified": false,\n    "admitted_as_full_s07_oof": false,\n    "evidence": [\n      "code cell 3: atlas_scores loads training_config and adapters.safetensors from fixed bundle, no fold-specific routing"\n    ],\n    "checkpoint_path": "datasets/josephadamski91/ntl-atlas-qwen3-8b-v1/atlas_adapter/atlas_adapter",\n    "dataset_display_name": "NTL ATLAS Qwen3 8B v1",\n    "recorded_ui_dataset_version": 3,\n    "recorded_ui_version_value": "22180590",\n    "base_model_path": "models/qwen-lm/qwen-3/transformers/8b-base/1",\n    "blocker": "Source-bundle inference and input hashes do not verify exclusion of official/external scoring prompts. Need adapter/pool training manifests or refitting."\n  },\n  "postprocessing": [\n    {\n      "id": "post.main_comp_mixture",\n      "kind": "probability_mixture",\n      "settings": {\n        "comp_mix": "0.15",\n        "atlas_mix": "0.10 where selected",\n        "pre_comp_division": "number of active nonzero LP members when sk_div=1"\n      },\n      "classification": "NO_CURRENT_FIT_BUT_UPSTREAM_EXCLUSION_REQUIRED",\n      "note": "Different test grouping/active branches can alter the pre-mixture member count. All contributing supervised paths must exclude the whole scoring batch."\n    },\n    {\n      "id": "post.sinkhorn",\n      "kind": "batch_coupled_probability_marginal_balancing",\n      "settings": {\n        "alpha": "1.0",\n        "iterations": 100,\n        "activation": "N>=18",\n        "column_mass_target": "N/18"\n      },\n      "classification": "NO_SUPERVISED_FIT_BUT_BATCH_COUPLED",\n      "note": "Uniform expected probability column mass, not guaranteed equal final label counts. Pooled mixed-source-fold OOF can leak supervised peer information through this operation."\n    },\n    {\n      "id": "post.typography_guards",\n      "kind": "hardcoded_label_conditioned_allowed_sets",\n      "classification": "RULE_DERIVATION_PROMPT_EXCLUSION_UNVERIFIED",\n      "note": "Hardcoded source allowed-label sets reportedly derived from train/matched-pool typography. No fold-local rule-derivation proof is supplied. Exclude from a balancing-only reduced screen or freeze a justified identical handling before evaluation."\n    },\n    {\n      "id": "post.dense_group_hungarian",\n      "kind": "test_group_coupled_injective_assignment",\n      "settings": {\n        "dense_group_size": "12..18"\n      },\n      "classification": "NO_CURRENT_FIT_BUT_BATCH_COUPLED",\n      "note": "Source assumes injective class assignments in inferred dense groups; all upstream scoring-batch prompt exclusions must hold."\n    }\n  ],\n  "ten_unconditional_positive_weight_complement_members": [\n    "stylegram",\n    "stylometry_lgb",\n    "debv3l",\n    "electl",\n    "desk",\n    "lora24_seed7",\n    "lora24_claude",\n    "lora20_q25_instyle",\n    "tokattn_deb",\n    "ultrabert"\n  ],\n  "conditional_complement_evidence": {\n    "frozen_centered_twins": [\n      "q3_8b",\n      "q25_7b",\n      "q3_4b",\n      "glm9b",\n      "debv3l"\n    ],\n    "gaussian_heads": [\n      "gauss_q3_8b_L22",\n      "gauss_glm9b_L8",\n      "gauss_q3_4b_L4",\n      "gauss_debv3l_L24"\n    ],\n    "adapted_centered_twins": [\n      "lora24_claude"\n    ],\n    "note": "Ten unconditional positive-weight contributions are not the full count of production evidence paths. Conditional grouped paths may be inactive on singleton preview rows."\n  },\n  "historical_input_receipt": {\n    "cpu_version": 8,\n    "script_version_id": 363603066,\n    "raw_report_sha256": "dd38302b9f4c135cd2a0d632e9af52c74dbb3f2c847ca2c3aa09dbe4d5a45db2",\n    "input_checks_passed": 71,\n    "independent_review_checks_passed": 20,\n    "serialized_torch_head_schemas_inspected": 37,\n    "scope": "Input paths/schema/content hashes; not holdout training provenance or hidden execution proof.",\n    "source_has_named_meta_missing_warnings": 37\n  },\n  "recorded_input_binding_versions": {\n    "source": "results/stage_12/DATASET_PIN_RECONCILIATION_20261029.json",\n    "dataset_pin_count": 14,\n    "pins": [\n      {\n        "display_name": "prvsiyan\'s Extended Name That LLM Dataset",\n        "selected_version_label": "Version 6 - Licence clarification: dataset CC BY 4.0 as a whole; only the 150 reproduced train_regen prompt texts in prompts.csv stay CC BY-SA 4.0 (pjmathematician). README/changelog updated.",\n        "ui_version_value": "22269045"\n      },\n      {\n        "display_name": "NTL free-proxy GPT and Gemini generations",\n        "selected_version_label": "Version 8 - Add prompt-bank-v2 batches v2b06-v2b10 (1,197 passages; now 5,452 passages on 1,344 prompts)",\n        "ui_version_value": "22401889"\n      },\n      {\n        "display_name": "Name That LLM: extra GPT and Gemini generations",\n        "selected_version_label": "Version 2 - batch 3: Gemini 3.1 Pro + GPT-6 Astra",\n        "ui_version_value": "22177031"\n      },\n      {\n        "display_name": "Name That LLM: free-proxy generations",\n        "selected_version_label": "Version 1 - Initial release",\n        "ui_version_value": "22010600"\n      },\n      {\n        "display_name": "Name That LLM: Qwen3-8B LoRA + Claude Head",\n        "selected_version_label": "Version 1 - Initial release",\n        "ui_version_value": "21994138"\n      },\n      {\n        "display_name": "NTL two-stage LoRA checkpoints and public pool",\n        "selected_version_label": "Version 1 - Initial release",\n        "ui_version_value": "22100594"\n      },\n      {\n        "display_name": "NTL two-stage LoRA part two",\n        "selected_version_label": "Version 1 - Initial release",\n        "ui_version_value": "22292189"\n      },\n      {\n        "display_name": "NTL Two-Stage LoRA24 Adapters (120k pool)",\n        "selected_version_label": "Version 1 - Initial release",\n        "ui_version_value": "22377778"\n      },\n      {\n        "display_name": "UltraBERT-380M Name That LLM 5-fold",\n        "selected_version_label": "Version 1 - Initial release",\n        "ui_version_value": "21895589"\n      },\n      {\n        "display_name": "Name That LLM: Extra API Passages",\n        "selected_version_label": "Version 2 - v1.1: same data; README.md synced with the corrected description (fixes broken characters and a cut-off description); file and column descriptions",\n        "ui_version_value": "21981458"\n      },\n      {\n        "display_name": "nlm-offline-wheels",\n        "selected_version_label": "Version 2 - add peft",\n        "ui_version_value": "22022378"\n      },\n      {\n        "display_name": "nlm-matched-clean",\n        "selected_version_label": "Version 3 - add pool_v5: 120k public passages (122k clean pool + own generations, mark + train-leak filters)",\n        "ui_version_value": "22318518"\n      },\n      {\n        "display_name": "NTL ATLAS Qwen3 8B v1",\n        "selected_version_label": "Version 3 - Add CC BY-SA 4.0 license for adapter weights and package documentation, Apache 2.0 license for bundled Python code, and credits. Model weights, inference code and original files are unchanged.",\n        "ui_version_value": "22180590"\n      },\n      {\n        "display_name": "Identify the model behind text",\n        "selected_version_label": "Version 7 - add seed-7 Qwen3-8B two-stage adapters (max_len 512; 5 folds + full)",\n        "ui_version_value": "22233098"\n      }\n    ],\n    "kernel_version_ids": [\n      358683059,\n      358745836,\n      358758062,\n      358770408,\n      358911769,\n      358928013,\n      358948428,\n      359652568\n    ],\n    "model_instance_version_ids": [\n      104433,\n      166254,\n      166265,\n      363135,\n      363139,\n      363140,\n      363149,\n      363153\n    ],\n    "mapping_limit": "UI version radio values are not assumed equivalent to saved notebook metadata sourceId values; exact per-path mapping is only supplied above where source evidence identifies it."\n  },\n  "proxy_feature_admission_leads": [\n    {\n      "id": "proxy.glm_frozen_cache",\n      "historical_status": "VERIFIED_FEATURE_CACHE_NOT_CLASSIFIER_OOF",\n      "source_script_version_id": 359405443,\n      "backbone_source_kernel_version_id": 358683059,\n      "path": "glm_features.npz",\n      "sha256": "2b41585a9482cef825962a8cc096697933c0b1237e8ede0712d1f8ab361d9414",\n      "shape": [\n        2699,\n        3,\n        4096\n      ],\n      "layers": [\n        8,\n        3,\n        24\n      ],\n      "dtype": "float32",\n      "role": "L8+L3 classifier features; L24 grouping geometry",\n      "fits_performed_by_feature_extraction": 0,\n      "recorded_order_and_finiteness_verified": true,\n      "fresh_mounted_gate_passed": false,\n      "new_fold_local_readout_required": true,\n      "common_fold_readout_ready": false,\n      "eligibility": "Best-prepared feature admission lead; requires actual bytes/hash/alignment/source checks then separately frozen fold-local fits. Adding existing GLM evidence is reference coverage, not a new S07 gain."\n    },\n    {\n      "id": "proxy.pretrained_deberta_cache",\n      "historical_status": "VERIFIED_FEATURE_CACHE_WITH_QUANTIZATION_AND_METADATA_LIMITS",\n      "source_script_version_id": 358745836,\n      "path": "enc_debv3l2_train.npz",\n      "sha256": "11eed58762ad128f97457dde9cc4ab03c7932e1b4b04845ec6664c834b1b1dda",\n      "role": "Pretrained DeBERTa mean L24+L6 for new common-fold supervised readouts; not main.deb fine-tuned checkpoint",\n      "dtype": "historical fp16 cache",\n      "fresh_mounted_gate_passed": false,\n      "new_fold_local_readout_required": true,\n      "eligibility": "Use only after actual safe alignment/schema/provenance admission. Object metadata must not be unpickled or bypassed; unknown alignment blocks admission."\n    },\n    {\n      "id": "proxy.text_style",\n      "historical_status": "RECIPE_AND_PRIOR_FOLD_LOCAL_OOF_EVIDENCE",\n      "role": "Deterministic style features and fold-local vectorizers/readouts rebuilt from official training data",\n      "fresh_fold_local_fit_completed": false,\n      "eligibility": "Can be rebuilt in common folds after exact implementation/parameters/context freeze; source all-row fitted transforms or external-pool rows cannot be reused as held-out."\n    }\n  ],\n  "historical_heldout_evidence": [\n    {\n      "id": "historical.lora1_oof",\n      "source_script_version_id": 358831649,\n      "evidence_path": "results/stage_06/LOra_COMPONENT_PROVENANCE.json",\n      "status": "VERIFIED_STANDALONE_PUBLIC_COMPONENT_OOF",\n      "exclusion_proof": "Recorded five outer fits each exclude 30 prompts, complete 2699 rows, no row/prompt overlap; saved numeric artifacts reconstructed.",\n      "limit": "Not an active S07 lora24_public checkpoint and not full-S07 OOF; source folds differ from other banks."\n    },\n    {\n      "id": "historical.seed7_lora_oof",\n      "source_script_version_id": 358911769,\n      "evidence_path": "results/stage_06b/component_audits.json",\n      "status": "RECONSTRUCTED_STANDALONE_HELDOUT_READOUT_OUTPUTS",\n      "limit": "Separate outer cache only; production uses job_final. Different fold scheme cannot be pooled for batch-coupled evaluation without whole-batch exclusion."\n    },\n    {\n      "id": "historical.token_attention_oof",\n      "source_script_version_ids": [\n        358928013,\n        358948428\n      ],\n      "evidence_path": "results/stage_06b/component_audits.json",\n      "status": "SOURCE_SEED0_PROMPT_DISJOINT_NUMERIC_OOF_VERIFIED",\n      "limit": "Training was not rerun; outer token-state/head banks unavailable for independent re-inference. Final head != OOF bank."\n    },\n    {\n      "id": "historical.stage06e_proxy",\n      "source_script_version_id": 359514801,\n      "evidence_path": "results/stage_06e/REPORT.md",\n      "status": "VALID_RAW_AVAILABLE_COMPONENT_PROXY_WITH_REUSED_DEVELOPMENT_PROMPTS",\n      "mean_balanced_accuracy": "0.548122 (historical raw proxy)",\n      "limit": "Seven branches, not S07. Report explicitly excludes balanced diagnostic from selection due to mixed source-fold peer supervision. Common-fold refitting required for coupled balancing."\n    }\n  ],\n  "common_fold_admission_requirements": [\n    "For each supervised adapter/norm/readout/preprocessor/calibrator and any external data, verify exclusion of every scoring-batch prompt, not only the current row.",\n    "Retain all same-prompt responses together and ensure all learned transforms fit only the outer training side; unlabelled test grouping must follow its declared protocol.",\n    "Use exact cached bytes/IDs/classes/fold mappings; no unpickle or fallback row-order assumptions.",\n    "Keep reused canonical folds labelled exploratory, not an untouched final holdout.",\n    "Do not evaluate balancing on pooled incompatible supervised OOF banks. Use common folds/refits from eligible frozen-base features or block the test.",\n    "Freeze reference members, fitting settings, context/stress construction and numerical fixtures before any candidate score.",\n    "Keep raw row IDs/passages/predictions/arrays/weights private; publish aggregate provenance and checks only."\n  ],\n  "no_raw_rows_or_predictions_exported": true,\n  "no_model_fit_or_inference_performed": true,\n  "no_gpu_or_paid_service_used": true,\n  "new_development_score": null,\n  "evidence_references": {\n    "source": {\n      "path": "notebooks/Stage_12_Public_651_Reproduction.ipynb",\n      "git_freeze": "55cf8622c592690c98f67b65c3e9b6381a726b32",\n      "upstream_script_version_id": 362013743,\n      "own_production_script_version_id": 363626171\n    },\n    "cpu": {\n      "path": "results/stage_12/CPU_V8_INDEPENDENT_ACTUAL_REVIEW.json",\n      "script_version_id": 363603066,\n      "scope": "Input/content admission, not training exclusion"\n    },\n    "coverage": {\n      "path": "results/stage_06g/REPRESENTATION_COVERAGE.md"\n    },\n    "e": {\n      "path": "results/stage_06e/REPORT.md",\n      "script_version_id": 359514801\n    },\n    "pin": {\n      "path": "results/stage_12/DATASET_PIN_RECONCILIATION_20261029.json"\n    }\n  }\n}\n'
assert hashlib.sha256(_runner_source.encode('utf-8')).hexdigest() == '4b30488d126bc6213d8e1d128d50d7f0641b33e90b13192af55abe413ae49629'
assert hashlib.sha256(_ledger_source.encode('utf-8')).hexdigest() == 'a4ad782567010180bebbe5bec81442560f3a619fcb712d3b5848926c9f1bb8f4'
_runner_path = Path('/kaggle/working/stage13a_cache_admission.py')
_runner_path.write_text(_runner_source, encoding='utf-8')
Path('/kaggle/working/stage13a_static_provenance.json').write_text(_ledger_source, encoding='utf-8')
print('Reviewed runner and static ledger hashes verified; no models loaded.')


## Run the admission audit

A failed contract stops this notebook. Output contains aggregate checks and hashes only. A report-writing failure is also a failure.


In [ ]:
subprocess.run([sys.executable, str(_runner_path), '--input-root', '/kaggle/input', '--output-root', '/kaggle/working/stage13a_admission'], check=True)


## Read the actual saved result

The static ledger records source review before execution. The result below is read from the actual newly written audit report. Its feature-admission status is not an OOF result or an accuracy gain.


In [ ]:
_report_path = Path('/kaggle/working/stage13a_admission/stage13a_input_admission.json')
_actual = json.loads(_report_path.read_text(encoding='utf-8'))
assert _actual['status'] == 'PASS_FROZEN_BASE_FEATURE_ADMISSION_ONLY'
assert _actual['scope'] == 'frozen_base_feature_admission_only'
assert _actual['contract_is_production'] is True and _actual['report_written'] is True
for _flag in ['fit_performed','model_inference_performed','balancing_performed',
              'held_out_predictions_produced','development_reference_admitted','full_s07_oof_admitted',
              'stage13b_batch_coupled_comparison_admitted','official_score_estimated']:
    assert _actual[_flag] is False
print('STAGE13A_SAVED_REPORT_READBACK', _actual['status'])
print(json.dumps({k: _actual[k] for k in ['rows','prompts','feature_shape','layers','checks_passed_count','runtime_versions','eligible_for_future_fold_local_readout_fitting']}, sort_keys=True))


## Next decision

If the feature gate passes, freeze the exact common-prompt-fold readout protocol before fitting. Full-fit S07 checkpoints and incompatible source-fold predictions remain excluded from claims of clean batch-coupled validation. Stage13B and any official submission remain separate decisions.
